過学習対策のテスト

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.ops.stochastic_depth import StochasticDepth
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import math
from collections import OrderedDict

# ==========================================
# 1. 設定：パスとパラメータ
# ==========================================
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/26_0413_clean'

# Scalerを再現するためにTrainデータも読み込みます
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'

# ★ 評価したいテストデータのCSVパスを指定してください
TEST_CSV_PATH  = f'{pre}/src/FF/AFF/dataset_upsamp/test_fixed.csv' # ※実際のパスに合わせて変更してください
name = "ConvNext_100epoch_custom_cnn"
# ★ 学習済みのモデル（重み）のパスを指定してください
MODEL_PATH = f"../save/OFtest/{name}/best_fusion_model.pth"
SAVE_DIR = f"../save/OFtest/{name}"

BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CNN_INNER_DROPOUT = 0.2

# ==========================================
# 2. モデルのクラス定義 (学習時と完全同一)
# ==========================================
class LayerNorm2d(nn.LayerNorm):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 3, 1)
        x = nn.functional.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        x = x.permute(0, 3, 1, 2)
        return x

class Permute(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.dims = dims
    def forward(self, x):
        return torch.permute(x, self.dims)

class CNBlock(nn.Module):
    def __init__(self, dim, layer_scale: float, stochastic_depth_prob: float, dropout_p: float):
        super().__init__()
        self.block = nn.Sequential(OrderedDict([
            ("0", nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim, bias=True)),
            ("1", Permute([0, 2, 3, 1])),
            ("2", nn.LayerNorm(dim, eps=1e-6)),
            ("3", nn.Linear(dim, 4 * dim, bias=True)),
            ("4", nn.GELU()),
            ("custom_drop", nn.Dropout(p=dropout_p)),
            ("5", nn.Linear(4 * dim, dim, bias=True)),
        ]))
        self.layer_scale = nn.Parameter(torch.ones(dim, 1, 1) * layer_scale)
        self.stoch_depth = StochasticDepth(stochastic_depth_prob, "row")

    def forward(self, input):
        result = self.block(input)
        result = result.permute(0, 3, 1, 2)
        result = self.layer_scale * result
        result = self.stoch_depth(result)
        result += input
        return result

def create_custom_convnext_features(pretrained=False, dropout_p=0.3):
    block_setting = [3, 3, 9, 3]
    dims = [96, 192, 384, 768]
    stoch_depth_prob = 0.1 
    features = nn.Sequential()
    features.add_module("0", nn.Sequential(
        nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
        LayerNorm2d(dims[0], eps=1e-6)
    ))
    curr_stage = 0
    dp_rates = [x.item() for x in torch.linspace(0, stoch_depth_prob, sum(block_setting))]
    for i in range(4):
        if i > 0:
            features.add_module(str(2 * i), nn.Sequential(
                LayerNorm2d(dims[i-1], eps=1e-6),
                nn.Conv2d(dims[i-1], dims[i], kernel_size=2, stride=2)
            ))
        stage_blocks = []
        for j in range(block_setting[i]):
            stage_blocks.append(CNBlock(dims[i], 1e-6, dp_rates[curr_stage], dropout_p))
            curr_stage += 1
        features.add_module(str(2 * i + 1), nn.Sequential(*stage_blocks))
    return features

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, initial_weight=0.5):
        super(CBAM, self).__init__()
        reduced_channels = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()
        self._initialize_weights(initial_weight)

    def _initialize_weights(self, target_prob):
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.conv_spatial.bias, logit_value)
            self.conv_spatial.weight.data *= 0.01 

    def forward(self, x):
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        ch_att = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        
        avg_out_sp = torch.mean(x, dim=1, keepdim=True)
        max_out_sp = torch.amax(x, dim=1, keepdim=True)
        sp_att = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out_sp, max_out_sp], dim=1)))
        return ch_att * sp_att

class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        # 評価時は pretrained=False でOK (保存された重みで上書きするため)
        self.dl_extractor = create_custom_convnext_features(pretrained=False, dropout_p=CNN_INNER_DROPOUT)
        dl_out_channels = 768
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2)
        )
        self.dl_norm = nn.BatchNorm2d(dl_out_channels)
        self.fusion_attention = CBAM(dl_out_channels, initial_weight=0.5) 
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels),
            nn.Dropout(p=0.3),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        Y = self.dl_extractor(img)
        Y = self.dl_norm(Y)
        B, C, H, W = Y.shape
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        weights = self.fusion_attention(X + Y)
        fused = weights * X + (1 - weights) * Y
        return self.classifier(fused)

# ==========================================
# 3. Datasetクラス
# ==========================================
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try: 
            image = Image.open(img_path).convert('RGB')
        except: 
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform: 
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# ==========================================
# 4. メイン評価関数
# ==========================================
def evaluate_model():
    print(f"Using Device: {DEVICE}")
    
    # テスト時のTransform (データ拡張は行わず、リサイズと正規化のみ)
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # 1. 学習データを読み込み、学習時と同じScalerを作成する
    print("Fitting Scaler using training data...")
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    train_ds = CustomMultiModalDataset(train_df, IMG_ROOT, transform=test_transform, is_train=True)
    trained_scaler = train_ds.scaler
    
    # # 2. テストデータを読み込み、作成したScalerを適用する
    # print(f"Loading Test Dataset from: {TEST_CSV_PATH}")
    # test_df = pd.read_csv(TEST_CSV_PATH)
    # test_ds = CustomMultiModalDataset(test_df, IMG_ROOT, scaler=trained_scaler, transform=test_transform, is_train=False)
    # test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    
    # 3. モデルの準備と重みのロード
    print("Loading Model...")
    model = FusionModel(num_classes=3).to(DEVICE)
    
    if os.path.exists(MODEL_PATH):
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        print(f"Successfully loaded weights from {MODEL_PATH}")
    else:
        print(f"Error: Model weights not found at {MODEL_PATH}")
        return

    # ★ 評価モードに切り替え (Dropout無効化等)
    model.eval()

    all_preds = []
    all_labels = []
    
    print("Starting Evaluation...")
    with torch.no_grad():
        for images, hc_feats, labels in test_loader:
            images = images.to(DEVICE)
            hc_feats = hc_feats.to(DEVICE)
            labels = labels.to(DEVICE)
            
            outputs = model(images, hc_feats)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # --- 評価結果の計算と表示 ---
    acc = accuracy_score(all_labels, all_preds)
    print("\n" + "="*50)
    print(f"Test Accuracy: {acc:.4f}")
    print("="*50)
    
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=["Class 0", "Class 1", "Class 2"]))
    
    # 混同行列の描画と保存
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=["Class 0", "Class 1", "Class 2"], 
                yticklabels=["Class 0", "Class 1", "Class 2"])
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    
    cm_save_path = os.path.join(SAVE_DIR, "confusion_matrix.png")
    plt.savefig(cm_save_path)
    plt.close()
    
    print(f"\nConfusion matrix saved to: {cm_save_path}")

if __name__ == "__main__":
    evaluate_model()

SyntaxError: 'return' outside function (1736527146.py, line 297)

In [4]:
import os
import glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.ops.stochastic_depth import StochasticDepth
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import math
from collections import OrderedDict

# ==========================================
# 1. 設定：パスとパラメータ
# ==========================================
pre = "/home/hiromu/energy"
IMG_ROOT = f'{pre}/data/26_0413_clean'
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp/train_max_upsampled.csv'
TEST_CSV_PATH  = f'{pre}/src/FF/AFF/dataset_upsamp/test_fixed.csv'

# ★ すべてのモデルフォルダが入っている大元のディレクトリ
BASE_SAVE_DIR = f'{pre}/src/FF/AFF/save/OFtest'

BATCH_SIZE = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CNN_INNER_DROPOUT = 0.2

# ==========================================
# 2. モデルのクラス定義 (変更なし)
# ==========================================
class LayerNorm2d(nn.LayerNorm):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 3, 1)
        x = nn.functional.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        x = x.permute(0, 3, 1, 2)
        return x

class Permute(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.dims = dims
    def forward(self, x):
        return torch.permute(x, self.dims)

class CNBlock(nn.Module):
    def __init__(self, dim, layer_scale: float, stochastic_depth_prob: float, dropout_p: float):
        super().__init__()
        self.block = nn.Sequential(OrderedDict([
            ("0", nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim, bias=True)),
            ("1", Permute([0, 2, 3, 1])),
            ("2", nn.LayerNorm(dim, eps=1e-6)),
            ("3", nn.Linear(dim, 4 * dim, bias=True)),
            ("4", nn.GELU()),
            ("custom_drop", nn.Dropout(p=dropout_p)),
            ("5", nn.Linear(4 * dim, dim, bias=True)),
        ]))
        self.layer_scale = nn.Parameter(torch.ones(dim, 1, 1) * layer_scale)
        self.stoch_depth = StochasticDepth(stochastic_depth_prob, "row")

    def forward(self, input):
        result = self.block(input)
        result = result.permute(0, 3, 1, 2)
        result = self.layer_scale * result
        result = self.stoch_depth(result)
        result += input
        return result

def create_custom_convnext_features(pretrained=False, dropout_p=0.3):
    block_setting = [3, 3, 9, 3]
    dims = [96, 192, 384, 768]
    stoch_depth_prob = 0.1 
    features = nn.Sequential()
    features.add_module("0", nn.Sequential(
        nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
        LayerNorm2d(dims[0], eps=1e-6)
    ))
    curr_stage = 0
    dp_rates = [x.item() for x in torch.linspace(0, stoch_depth_prob, sum(block_setting))]
    for i in range(4):
        if i > 0:
            features.add_module(str(2 * i), nn.Sequential(
                LayerNorm2d(dims[i-1], eps=1e-6),
                nn.Conv2d(dims[i-1], dims[i], kernel_size=2, stride=2)
            ))
        stage_blocks = []
        for j in range(block_setting[i]):
            stage_blocks.append(CNBlock(dims[i], 1e-6, dp_rates[curr_stage], dropout_p))
            curr_stage += 1
        features.add_module(str(2 * i + 1), nn.Sequential(*stage_blocks))
    return features

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, initial_weight=0.5):
        super(CBAM, self).__init__()
        reduced_channels = max(1, channels // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid_channel = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid_spatial = nn.Sigmoid()
        self._initialize_weights(initial_weight)

    def _initialize_weights(self, target_prob):
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.conv_spatial.bias, logit_value)
            self.conv_spatial.weight.data *= 0.01 

    def forward(self, x):
        avg_out = self.fc(torch.mean(x, dim=(2, 3)))
        max_out = self.fc(torch.amax(x, dim=(2, 3)))
        ch_att = self.sigmoid_channel(avg_out + max_out).unsqueeze(-1).unsqueeze(-1)
        
        avg_out_sp = torch.mean(x, dim=1, keepdim=True)
        max_out_sp = torch.amax(x, dim=1, keepdim=True)
        sp_att = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out_sp, max_out_sp], dim=1)))
        return ch_att * sp_att

class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        self.dl_extractor = create_custom_convnext_features(pretrained=False, dropout_p=CNN_INNER_DROPOUT)
        dl_out_channels = 768
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2)
        )
        self.dl_norm = nn.BatchNorm2d(dl_out_channels)
        self.fusion_attention = CBAM(dl_out_channels, initial_weight=0.5) 
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels),
            nn.Dropout(p=0.3),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        Y = self.dl_extractor(img)
        Y = self.dl_norm(Y)
        B, C, H, W = Y.shape
        X = self.hc_projector(hc_vec).view(B, C, 1, 1).expand(B, C, H, W)
        weights = self.fusion_attention(X + Y)
        fused = weights * X + (1 - weights) * Y
        return self.classifier(fused)

# ==========================================
# 3. Datasetクラス (変更なし)
# ==========================================
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler=None, transform=None, is_train=True):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        features = self.df[self.feature_cols].values
        
        if is_train:
            self.scaler = StandardScaler()
            self.hc_features = self.scaler.fit_transform(features).astype('float32')
        else:
            assert scaler is not None
            self.scaler = scaler
            self.hc_features = self.scaler.transform(features).astype('float32')
            
        self.labels = self.df['Label'].values

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_root, 'mask', row['category'], row['filename'])
        try: 
            image = Image.open(img_path).convert('RGB')
        except: 
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform: 
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, hc_feat, label

# ==========================================
# 4. 全モデルの一括評価関数
# ==========================================
def evaluate_all_models():
    print(f"Using Device: {DEVICE}")
    
    # テスト時のTransform
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # 共通部分の初期化 (1回だけで済む処理)
    print("Fitting Scaler using training data...")
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    train_ds = CustomMultiModalDataset(train_df, IMG_ROOT, transform=test_transform, is_train=True)
    trained_scaler = train_ds.scaler
    
    print(f"Loading Test Dataset from: {TEST_CSV_PATH}")
    test_df = pd.read_csv(TEST_CSV_PATH)
    test_ds = CustomMultiModalDataset(test_df, IMG_ROOT, scaler=trained_scaler, transform=test_transform, is_train=False)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    
    # モデルインスタンスを1つだけ作成
    model = FusionModel(num_classes=3).to(DEVICE)
    model.eval()

    # BASE_SAVE_DIR 内のすべてのフォルダを取得
    if not os.path.exists(BASE_SAVE_DIR):
        print(f"Error: Directory not found -> {BASE_SAVE_DIR}")
        return

    folders = [f.path for f in os.scandir(BASE_SAVE_DIR) if f.is_dir()]
    
    if not folders:
        print(f"No folders found in {BASE_SAVE_DIR}")
        return

    print(f"\nFound {len(folders)} folders. Starting evaluation...")

    for folder_path in folders:
        model_path = os.path.join(folder_path, "best_fusion_model.pth")
        
        # モデルファイルが存在しないフォルダはスキップ
        if not os.path.exists(model_path):
            print(f"Skipping {os.path.basename(folder_path)}: 'best_fusion_model.pth' not found.")
            continue
            
        print("\n" + "="*60)
        print(f"Evaluating Model: {os.path.basename(folder_path)}")
        print("="*60)
        
        # 重みをロード
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        
        all_preds = []
        all_labels = []
        
        # テスト実行
        with torch.no_grad():
            for images, hc_feats, labels in test_loader:
                images = images.to(DEVICE)
                hc_feats = hc_feats.to(DEVICE)
                labels = labels.to(DEVICE)
                
                outputs = model(images, hc_feats)
                _, preds = torch.max(outputs, 1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        # 評価指標の計算
        acc = accuracy_score(all_labels, all_preds)
        cls_report = classification_report(all_labels, all_preds, target_names=["Class 0", "Class 1", "Class 2"])
        
        print(f"Test Accuracy: {acc:.4f}")
        
        # 1. テキストレポートの保存
        report_save_path = os.path.join(folder_path, "evaluation_report.txt")
        with open(report_save_path, "w", encoding="utf-8") as f:
            f.write(f"Test Accuracy: {acc:.4f}\n\n")
            f.write("Classification Report:\n")
            f.write(cls_report)
        
        # 2. 混同行列画像の保存
        cm = confusion_matrix(all_labels, all_preds)
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=["Class 0", "Class 1", "Class 2"], 
                    yticklabels=["Class 0", "Class 1", "Class 2"])
        plt.title(f'Confusion Matrix ({os.path.basename(folder_path)})')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        
        cm_save_path = os.path.join(folder_path, "confusion_matrix.png")
        plt.savefig(cm_save_path)
        plt.close()
        
        print(f"-> Saved evaluation_report.txt")
        print(f"-> Saved confusion_matrix.png")

    print("\nAll evaluations finished successfully!")

if __name__ == "__main__":
    evaluate_all_models()

Using Device: cuda
Fitting Scaler using training data...
Loading Test Dataset from: /home/hiromu/energy/src/FF/AFF/dataset_upsamp/test_fixed.csv

Found 13 folders. Starting evaluation...

Evaluating Model: ConvNext_100epoch_fixed_dataset_AdamW+shcedul
Test Accuracy: 0.5486
-> Saved evaluation_report.txt
-> Saved confusion_matrix.png

Evaluating Model: -scheD2,2,2B8
Test Accuracy: 0.6042
-> Saved evaluation_report.txt
-> Saved confusion_matrix.png

Evaluating Model: -scheD2,3,4B32
Test Accuracy: 0.6111
-> Saved evaluation_report.txt
-> Saved confusion_matrix.png

Evaluating Model: +scheD2,3,4B8
Test Accuracy: 0.6319
-> Saved evaluation_report.txt
-> Saved confusion_matrix.png

Evaluating Model: -scheD3,2,3B8+col
Test Accuracy: 0.5833
-> Saved evaluation_report.txt
-> Saved confusion_matrix.png

Evaluating Model: -scheD2,2,2B4+col
Test Accuracy: 0.5694
-> Saved evaluation_report.txt
-> Saved confusion_matrix.png

Evaluating Model: -scheD2,3,4B16
Test Accuracy: 0.6042
-> Saved evaluation_

In [4]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.ops.stochastic_depth import StochasticDepth
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import math
from collections import OrderedDict

# --- 1. 設定: ファイルパスとパラメータ ---
pre = "/home/hiromu/energy"
# ★ テストに使用する画像のルートディレクトリ（適宜変更してください）
IMG_ROOT = f'{pre}/data/1201_humomentstest' 

# スケーラーを訓練時と同じ状態にするため、Train用のCSVも読み込みます
TRAIN_CSV_PATH = f'{pre}/src/FF/AFF/dataset_upsamp_clean/train_max_upsampled.csv'
TEST_CSV_PATH  = f'{pre}/src/FF/AFF/dataset_upsamp_clean/test_fixed.csv'

# 学習時に合わせた設定
TARGET_EPOCH = 100
CNN_INNER_DROPOUT = 0.2
SAVE_DIR = f"../save/OFtest/ConvNext_{TARGET_EPOCH}epoch_custom_cnn"
MODEL_WEIGHT_PATH = f'{SAVE_DIR}/best_fusion_model.pth'


# --- 2. モデルの定義 (ロード用に学習時と全く同じものを定義) ---

class LayerNorm2d(nn.LayerNorm):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 3, 1)
        x = nn.functional.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        x = x.permute(0, 3, 1, 2)
        return x

class Permute(nn.Module):
    def __init__(self, dims):
        super().__init__()
        self.dims = dims
    def forward(self, x):
        return torch.permute(x, self.dims)

class CNBlock(nn.Module):
    def __init__(self, dim, layer_scale: float, stochastic_depth_prob: float, dropout_p: float):
        super().__init__()
        self.block = nn.Sequential(OrderedDict([
            ("0", nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim, bias=True)),
            ("1", Permute([0, 2, 3, 1])),
            ("2", nn.LayerNorm(dim, eps=1e-6)),
            ("3", nn.Linear(dim, 4 * dim, bias=True)),
            ("4", nn.GELU()),
            ("custom_drop", nn.Dropout(p=dropout_p)),
            ("5", nn.Linear(4 * dim, dim, bias=True)),
        ]))
        self.layer_scale = nn.Parameter(torch.ones(dim, 1, 1) * layer_scale)
        self.stoch_depth = StochasticDepth(stochastic_depth_prob, "row")

    def forward(self, input):
        result = self.block(input)
        result = result.permute(0, 3, 1, 2)
        result = self.layer_scale * result
        result = self.stoch_depth(result)
        result += input
        return result

def create_custom_convnext_features(pretrained=False, dropout_p=0.3):
    block_setting = [3, 3, 9, 3]
    dims = [96, 192, 384, 768]
    stoch_depth_prob = 0.1 
    features = nn.Sequential()
    features.add_module("0", nn.Sequential(
        nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
        LayerNorm2d(dims[0], eps=1e-6)
    ))
    curr_stage = 0
    dp_rates = [x.item() for x in torch.linspace(0, stoch_depth_prob, sum(block_setting))]
    for i in range(4):
        if i > 0:
            features.add_module(str(2 * i), nn.Sequential(
                LayerNorm2d(dims[i-1], eps=1e-6),
                nn.Conv2d(dims[i-1], dims[i], kernel_size=2, stride=2)
            ))
        stage_blocks = []
        for j in range(block_setting[i]):
            stage_blocks.append(CNBlock(dims[i], 1e-6, dp_rates[curr_stage], dropout_p))
            curr_stage += 1
        features.add_module(str(2 * i + 1), nn.Sequential(*stage_blocks))
    return features

class ChannelAttentionGate1D(nn.Module):
    def __init__(self, channels, reduction=16, initial_prob=0.5):
        super().__init__()
        reduced_channels = max(1, channels // reduction)
        self.attention = nn.Sequential(
            nn.Linear(channels, reduced_channels),
            nn.ReLU(inplace=True),
            nn.Linear(reduced_channels, channels)
        )
        self.sigmoid = nn.Sigmoid()
        self._initialize_weights(initial_prob)

    def _initialize_weights(self, target_prob):
        logit_value = math.log(target_prob / (1 - target_prob))
        with torch.no_grad():
            nn.init.constant_(self.attention[2].bias, logit_value)
            self.attention[2].weight.data *= 0.01

    def forward(self, x, y):
        context = x + y
        weights = self.sigmoid(self.attention(context))
        return weights * x + (1 - weights) * y

class FusionModel(nn.Module):
    def __init__(self, num_classes):
        super(FusionModel, self).__init__()
        dl_out_channels = 768
        
        self.dl_extractor = create_custom_convnext_features(pretrained=False, dropout_p=CNN_INNER_DROPOUT)
        
        self.dl_pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(dl_out_channels)
        )
        
        self.hc_projector = nn.Sequential(
            nn.Linear(10, dl_out_channels),
            nn.BatchNorm1d(dl_out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2)
        )
        
        self.fusion_gate = ChannelAttentionGate1D(dl_out_channels, initial_prob=0.5) 
        
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(dl_out_channels, num_classes)
        )

    def forward(self, img, hc_vec):
        img_feat = self.dl_extractor(img)
        img_feat_1d = self.dl_pool(img_feat)
        hc_feat_1d = self.hc_projector(hc_vec)
        fused_feat = self.fusion_gate(hc_feat_1d, img_feat_1d)
        return self.classifier(fused_feat)


# --- 3. Datasetの定義 (マルチモーダル版) ---
class CustomMultiModalDataset(Dataset):
    def __init__(self, dataframe, img_root, scaler, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
        
        # 評価用なので、渡された学習済みscalerでtransformのみ行う
        features = self.df[self.feature_cols].values
        self.hc_features = scaler.transform(features).astype('float32')
        self.labels = self.df['Label'].values

    def __len__(self): 
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # ファイル名の '_mask' を '_combined' に置換
        original_filename = row['filename']
        actual_filename = original_filename.replace('_mask', '_combined')
        
        # 'combined' フォルダを参照
        img_path = os.path.join(self.img_root, 'combined', row['category'], actual_filename)
        
        try: 
            image = Image.open(img_path).convert('RGB')
        except Exception as e: 
            print(f"Error loading {img_path}: {e}")
            image = Image.new('RGB', (224, 224), (0, 0, 0))
            
        if self.transform: 
            image = self.transform(image)
            
        hc_feat = torch.tensor(self.hc_features[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        
        return image, hc_feat, label


# --- 4. メインテスト関数 ---
def evaluate_model():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if not os.path.exists(MODEL_WEIGHT_PATH):
        print(f"Error: Model weight not found at {MODEL_WEIGHT_PATH}\n先に学習を完了させてください。")
        return

    print("=== Starting Test Evaluation for Late Fusion Model ===")

    # 1. 訓練時のスケーラーを再現
    print("Fitting StandardScaler on Training data...")
    train_df = pd.read_csv(TRAIN_CSV_PATH)
    feature_cols = ['h0', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'size_count', 'R']
    scaler = StandardScaler()
    scaler.fit(train_df[feature_cols].values)

    # 2. テストデータセットの準備 (Validation時と同じ224x224のリサイズを使用)
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_df = pd.read_csv(TEST_CSV_PATH)
    test_dataset = CustomMultiModalDataset(test_df, IMG_ROOT, scaler=scaler, transform=test_transform)
    test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

    # 3. モデルの準備と重みのロード
    model = FusionModel(num_classes=3).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_WEIGHT_PATH, map_location=DEVICE))
    model.eval()
    print(f"Successfully loaded weights from {MODEL_WEIGHT_PATH}")

    all_preds, all_labels = [], []
    
    # 4. 推論ループ
    print("Running inference on test dataset...")
    with torch.no_grad():
        for images, hc_feats, labels in test_loader:
            images, hc_feats, labels = images.to(DEVICE), hc_feats.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images, hc_feats)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # 5. 結果の算出と表示
    accuracy = accuracy_score(all_labels, all_preds)
    print(f"\nOverall Test Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(all_labels, all_preds, digits=4))

    # 6. 混同行列の保存
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix (Late Fusion Model)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    
    os.makedirs(SAVE_DIR, exist_ok=True)
    cm_save_path = f'{SAVE_DIR}/test_confusion_matrix.png'
    plt.savefig(cm_save_path)
    plt.close()
    
    print(f"Confusion matrix saved to {cm_save_path}")
    print("Test Evaluation Finished.")

if __name__ == "__main__":
    evaluate_model()

=== Starting Test Evaluation for Late Fusion Model ===
Fitting StandardScaler on Training data...
Successfully loaded weights from ../save/OFtest/ConvNext_100epoch_custom_cnn/best_fusion_model.pth
Running inference on test dataset...

Overall Test Accuracy: 0.6901

Classification Report:
              precision    recall  f1-score   support

           0     0.6981    0.8043    0.7475        46
           1     0.6757    0.5208    0.5882        48
           2     0.6923    0.7500    0.7200        48

    accuracy                         0.6901       142
   macro avg     0.6887    0.6917    0.6852       142
weighted avg     0.6886    0.6901    0.6844       142

Confusion matrix saved to ../save/OFtest/ConvNext_100epoch_custom_cnn/test_confusion_matrix.png
Test Evaluation Finished.
